# Проба пера по Сбербанку

# проба пера по сбербанку отраоботать принципиальную возможность 
# получить хоть какие -то адеквартные и предиктивные данные

In [ ]:
#1
# Import libraries загружаем необходимые библиотеки 
import pandas as pd
import sys
import os
import warnings
warnings.filterwarnings("ignore")
import pandas_ta as ta
import seaborn as sns

# Рисовать графики сразу же
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier

plt.style.use('ggplot')  # Красивые графики
plt.rcParams['figure.figsize'] = (15, 5)  # Размер картинок

In [ ]:
#2
# https://github.com/bukosabino/ta/blob/master/examples_to_use/visualize_features.ipynb
# группа данных и библиотек для технического анализа
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.style.use('seaborn')

import ta

In [ ]:
#3
# пишем функцию для смещения будем двигать время назад 
# определим новую функцию Python с именем series_to_supervised() , 
# которая берет одномерный или многомерный временной ряд и оформляет его как набор данных обучения с учителем.
# взял вот отсюда https://machinelearningmastery.com/convert-time-series-supervised-learning-problem-python/
from pandas import DataFrame
from pandas import concat
import numpy as np

def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):
	"""
	Frame a time series as a supervised learning dataset.
	Arguments:
		data: Sequence of observations as a list or NumPy array.
        Последовательность наблюдений в виде списка или двумерного массива NumPy. Необходимый.
		n_in: Number of lag observations as input (X).
        Количество наблюдений запаздывания в качестве входных данных ( X ). 
        Значения могут быть между [1..len(data)] Необязательно. По умолчанию 1.
		n_out: Number of observations as output (y).
        Количество наблюдений в качестве вывода ( y ). 
        Значения могут быть между [0..len(data)-1]. Необязательный. По умолчанию 1.
		dropnan: Boolean whether or not to drop rows with NaN values.
        Логическое значение, следует ли удалять строки со значениями NaN. 
        Необязательный. По умолчанию Истина.
	Returns:
		Pandas DataFrame of series framed for supervised learning.
	"""
	n_vars = 1 if type(data) is list else data.shape[1]
	df = DataFrame(data)
	col_list = data.columns.tolist() # emaly add
	cols, names = list(), list()
	# input sequence (t-n, ... t-1)
	for i in range(n_in, 0, -1):
		cols.append(df.shift(i))
		names += [('%s_%d(t-%d)' % (col_list[j], j+1, i)) for j in range(n_vars)] # emaly modified
	# forecast sequence (t, t+1, ... t+n)
	for i in range(0, n_out):
		cols.append(df.shift(-i))
		if i == 0:
			names += [('%s_var%d(t)' % (col_list[j],j+1)) for j in range(n_vars)]# emaly modified
		else:
			names += [('%s_var%d(t+%d)' % (col_list[j], j+1, i)) for j in range(n_vars)]# emaly modified
	# put it all together
	agg = concat(cols, axis=1)
	agg.columns = names
	# drop rows with NaN values
	if dropnan:
		agg.dropna(inplace=True)
	return agg

In [ ]:
#4
# смотрим на версии с чем работаем, для понимания 
print('Python version ' + sys.version)
print('Pandas version: ' + pd.__version__)

In [ ]:
#5


#создание DataFrame с помощью чтения данных из файла
frame = pd.read_csv('SBER_210101_220424.csv', header=0, sep=';')
#frame = pd.read_csv('SBER_200101_220424.csv', header=0, sep=';',parse_dates=[['<DATE>', '<TIME>']])
# https://www.finam.ru/profile/moex-akcii/sberbank/export/ скачано вот тут 

In [ ]:
#6
frame.info()

In [ ]:
#7
frame.head()

In [ ]:
# Takes a dataframe and the column name as inputs
def create_datetime_features(df, c):
    if (df[c].dtype!='datetime64[ns]'):
        pd.to_datetime(df[c]) 
        
    df['year'] = df[c].dt.year
    df['month'] = df[c].dt.month
    df['day'] = df[c].dt.day
    df['hour'] = df[c].dt.hour
    df['session'] = df['hour'].apply(lambda x: get_session(x))
    df['minute'] = df[c].dt.minute
    df['second'] = df[c].dt.second
    df['week'] = df[c].dt.week
    df['dayofweek'] = df[c].dt.dayofweek
    df['isWeekend'] = (df[c].dt.dayofweek >=5).astype(int)
    df['isWeekday'] = (df[c].dt.dayofweek < 5).astype(int)
    
    df['dayofyear'] = df[c].dt.dayofyear
    df['quarter'] = df[c].dt.quarter
    df['isMonthStart'] = df[c].dt.is_month_start.astype(int)
    df['isMonthEnd'] = df[c].dt.is_month_end.astype(int)
    df['isQuarterStart'] = df[c].dt.is_quarter_start.astype(int)
    df['isQuarterEnd'] = df[c].dt.is_quarter_end.astype(int)
    df['isYearStart'] = df[c].dt.is_year_start.astype(int)
    df['isYearEnd'] = df[c].dt.is_year_end.astype(int)
    df['isLeapYear'] = df[c].dt.is_leap_year.astype(int)
    
    df["ta_rsi"] = df.ta.rsi(close='<CLOSE>')
    df["ta_rsx"] = df.ta.rsx(close='<CLOSE>')
    df["ta_eri"] = df.ta.eri(close='<CLOSE>')
    df['ta_stoch'] = df.ta.stoch(high='<HIGH>', low='<LOW>', close='<CLOSE>').iloc[:,0]
    df['ta_macd'] = df.ta.macd(close='<CLOSE>').iloc[:,0]
    df['ta_adx'] = df.ta.adx(high='<HIGH>', low='<LOW>', close='<CLOSE>').iloc[:,0]
    df['ta_atr'] = df.ta.atr(high='<HIGH>', low='<LOW>', close='<CLOSE>')
    df['ta_cci'] = df.ta.cci(high='<HIGH>', low='<LOW>', close='<CLOSE>')
    df['ta_cmo'] = df.ta.cmo(close='<CLOSE>')
    df['ta_coppock'] = df.ta.coppock(close='<CLOSE>')
    df['ta_dema'] = df.ta.dema(close='<CLOSE>')
    df['ta_dpo'] = df.ta.dpo(close='<CLOSE>')
    df['ta_er'] = df.ta.er(close='<CLOSE>')
    df['ta_fisher'] = df.ta.fisher(high='<HIGH>', low='<LOW>').iloc[:,0]
    df['ta_hma'] = df.ta.hma(close='<CLOSE>')
    df['ta_kc'] = df.ta.kc(high='<HIGH>', low='<LOW>', close='<CLOSE>').iloc[:,0]
    df['ta_kst'] = df.ta.kst(close='<CLOSE>').iloc[:,0]
    df['ta_linreg'] = df.ta.linreg(close='<CLOSE>')
    df['ta_log_return'] = df.ta.log_return(close='<CLOSE>')
    df['ta_midpoint'] = df.ta.midpoint(close='<CLOSE>')
    df['ta_midprice'] = df.ta.midprice(high='<HIGH>', low='<LOW>')
    df['ta_mom'] = df.ta.mom(close='<CLOSE>')
    df['ta_natr'] = df.ta.natr(high='<HIGH>', low='<LOW>', close='<CLOSE>')
    df['ta_nvi'] = df.ta.nvi(close='<CLOSE>', volume='<VOL>')
    df['ta_obv'] = df.ta.obv(close='<CLOSE>', volume='<VOL>')
    df['ta_ppo'] = df.ta.ppo(close='<CLOSE>').iloc[:,0]
    df['ta_psar'] = df.ta.psar(high='<HIGH>', low='<LOW>', close='<CLOSE>').iloc[:,0]
    df['ta_pvi'] = df.ta.pvi(close='<CLOSE>', volume='<VOL>')
    df['ta_pvol'] = df.ta.pvol(close='<CLOSE>', volume='<VOL>')
    df['ta_pvr'] = df.ta.pvr(close='<CLOSE>', volume='<VOL>')
    df['ta_pvt'] = df.ta.pvt(close='<CLOSE>', volume='<VOL>')
    df['ta_qqe'] = df.ta.qqe(close='<CLOSE>').iloc[:,0]
    df['ta_roc'] = df.ta.roc(close='<CLOSE>')
    df['ta_rsi'] = df.ta.rsi(close='<CLOSE>')
    df['ta_rvi'] = df.ta.rvi(close='<CLOSE>')
    df['ta_slope'] = df.ta.slope(close='<CLOSE>')
    df['ta_sma'] = df.ta.sma(close='<CLOSE>')
    df['ta_smi'] = df.ta.smi(close='<CLOSE>').iloc[:,0]
    df['ta_squeeze'] = df.ta.squeeze(high='<HIGH>', low='<LOW>', close='<CLOSE>').iloc[:,0]
    df['ta_stc'] = df.ta.stc(close='<CLOSE>')
    df['ta_stdev'] = df.ta.stdev(close='<CLOSE>')
    df['ta_stoch_rsi'] = df.ta.stochrsi(close='<CLOSE>').iloc[:,0]
    df['ta_supertrend'] = df.ta.supertrend(high='<HIGH>', low='<LOW>', close='<CLOSE>').iloc[:,0]
    df['ta_t3'] = df.ta.t3(close='<CLOSE>')
    df['ta_tema'] = df.ta.tema(close='<CLOSE>')
    df['ta_trix'] = df.ta.trix(close='<CLOSE>').iloc[:,0]
    df['ta_truerange'] = df.ta.true_range(high='<HIGH>', low='<LOW>', close='<CLOSE>')
    df['ta_tsi'] = df.ta.tsi(close='<CLOSE>').iloc[:,0]
    df['ta_ui'] = df.ta.ui(close='<CLOSE>')
    df['ta_uo'] = df.ta.uo(high='<HIGH>', low='<LOW>', close='<CLOSE>')
    df['ta_variance'] = df.ta.variance(close='<CLOSE>')
    df['ta_vhf'] = df.ta.vhf(close='<CLOSE>')
    df['ta_vidya'] = df.ta.vidya(close='<CLOSE>')
    df['ta_vp'] = df.ta.vp(close='<CLOSE>', volume='<VOL>').iloc[:,0]
    df['ta_vwap'] = df.ta.vwap(high='<HIGH>', low='<LOW>', close='<CLOSE>', volume='<VOL>')
    df['ta_vwma'] = df.ta.vwma(close='<CLOSE>', volume='<VOL>')
    df['ta_wcp'] = df.ta.wcp(high='<HIGH>', low='<LOW>', close='<CLOSE>')
    df['ta_wma'] = df.ta.wma(close='<CLOSE>')
    df['ta_zscore'] = df.ta.zscore(close='<CLOSE>')
    df['ta_zlma'] = df.ta.zlma(close='<CLOSE>')
    df['ta_willr'] = df.ta.willr(high ='<HIGH>', low= '<LOW>', close='<CLOSE>')
    df['ta_ttm_trend'] = df.ta.ttm_trend(high ='<HIGH>', low= '<LOW>', close='<CLOSE>')
    
    df['ta_true_range'] = df.ta.true_range(high ='<HIGH>', low= '<LOW>', close='<CLOSE>')

    df[["ta_bb_lower", "ta_bb_mid", "ta_bb_upper", "ta_bb_bandwidth", "ta_bb_percent"]] = df.ta.bbands(close='<CLOSE>')
    
    
def get_session(time):
    if time > 4 and time<12:
        return 0 # Morning
    elif time >= 12 and time < 17:
        return 1 # Afternoon
    elif time >= 17 and time < 21:
        return 2 # Evening
    else:
        return 3 # Night   

In [ ]:
frame

In [ ]:
#10
frame.drop(columns = ['<TICKER>','<PER>'],axis = 1, inplace=True)

In [ ]:
#11
frame.info()

In [ ]:
#12
frame.head()

In [ ]:
#13
fixed_df = frame[:10000]


In [ ]:
#14
fixed_df

In [ ]:
#15
type(fixed_df)

In [ ]:
#16
def ohlc_to_candlestick(conversion_array):
    candlestick_data = [0,0,0,0,0]

    if conversion_array['<CLOSE>']>conversion_array['<OPEN>']:
        candle_type=1
        wicks_up=conversion_array['<HIGH>']-conversion_array['<CLOSE>']
        wicks_down=conversion_array['<OPEN>']-conversion_array['<LOW>']
        body_size=conversion_array['<CLOSE>']-conversion_array['<OPEN>']
        all_size = conversion_array['<HIGH>']-conversion_array['<LOW>']
  
   

    else:
        candle_type=0
        wicks_up=conversion_array['<HIGH>']-conversion_array['<OPEN>']
        wicks_down=conversion_array['<CLOSE>']-conversion_array['<LOW>']
        body_size=conversion_array['<OPEN>']-conversion_array['<CLOSE>']
        all_size = conversion_array['<HIGH>']-conversion_array['<LOW>']
      
    candlestick_data[0]=candle_type
    candlestick_data[1]=wicks_up
    candlestick_data[2]=wicks_down
    candlestick_data[3]=body_size
    candlestick_data[4]=all_size

    return candlestick_data[0], candlestick_data[1], candlestick_data[2], candlestick_data[3], candlestick_data[4]


In [ ]:
#17
fixed_df['cndl_candle_type'],fixed_df['cndl_wicks_up'],fixed_df['cndl_wicks_down'],fixed_df['cndl_body_size'], fixed_df['cndl_all_size']= zip(*fixed_df.apply(ohlc_to_candlestick, axis=1)) 


In [ ]:
#18
fixed_df.head()

In [ ]:
#19
# конвертируем дату
# нам нужно из целочисленного формата получить дату
# формат finam YYMMDD
fixed_df['datetime'] = pd.to_datetime(fixed_df['<DATE>'].astype(str) + fixed_df['<TIME>'].astype(str).str.zfill(6), format='%y%m%d%H%M%S')
fixed_df = fixed_df.set_index('datetime')
fixed_df.head()

In [ ]:
#20
fixed_df.info()

In [ ]:
#21
# добавим технические индикаторы
create_datetime_features(fixed_df, 'datetime')

In [ ]:
#22
fixed_df.shape

In [ ]:
#23
fixed_df.head(3)

In [ ]:
#24
# визуализация цены закрытия
fixed_df['<CLOSE>'].plot(title='SBER Close Price')

In [ ]:
#25
# создаём целевую переменную - направление движения цены
# 1 - цена выросла, 0 - цена упала или осталась
fixed_df['target'] = (fixed_df['<CLOSE>'].shift(-1) > fixed_df['<CLOSE>']).astype(int)
fixed_df['target'].value_counts()

In [ ]:
#26
fixed_df.shape

In [ ]:
#27
# убираем NaN
fixed_df.dropna(inplace=True)
fixed_df.shape

In [ ]:
#28
# список признаков
feature_cols = [c for c in fixed_df.columns if c not in ['<DATE>', '<TIME>', 'target']]
print(f'Features count: {len(feature_cols)}')
feature_cols

In [ ]:
#29
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = fixed_df[feature_cols]
y = fixed_df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')

In [ ]:
#30
# обучаем модель
model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
model.fit(X_train_scaled, y_train)

In [ ]:
#31
from sklearn.metrics import classification_report, accuracy_score

y_pred = model.predict(X_test_scaled)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred))

In [ ]:
#32
# важность признаков
importances = pd.Series(model.feature_importances_, index=feature_cols)
importances.sort_values(ascending=False).head(20).plot(kind='barh', title='Feature Importances (Top 20)')
plt.tight_layout()
plt.show()